In [7]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\arbaa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Creating Spark Session

This code block will allow the spark session to start and configure spark to load the spark-xml package in order to read the XML wiki file.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("WikipediaWordCount")
    .config("spark.jar.packages", "com.databricks:spark-xml_2.12:0.18.0")
    .getOrCreate()
)

This code block creates a dataframe to read the xml file, then allows spark to block each '<page> ... </page>' in the file as a row in the dataframe. Lastly, the load function locates and loads the xml file from the data folder.

In [3]:
dataframe = (
    spark.read.format("xml")
    .option("rowTag", "page")
    .load("data\simplewiki-latest-pages-articles.xml\simplewiki-latest-pages-articles.xml")
)

In [14]:
spark

This selectExpr function allows me to display the sum of the pages in one colum under the name total_pages.

I was also able to show the structure of the dataframe by printing out the schema. This gives me a clearer idea of the layout of the data within the XML file.

In [4]:
dataframe.selectExpr("count(1) as total_pages").show()
dataframe.printSchema()

+-----------+
|total_pages|
+-----------+
|     536723|
+-----------+

root
 |-- id: long (nullable = true)
 |-- ns: long (nullable = true)
 |-- redirect: struct (nullable = true)
 |    |-- _title: string (nullable = true)
 |-- revision: struct (nullable = true)
 |    |-- comment: string (nullable = true)
 |    |-- contributor: struct (nullable = true)
 |    |    |-- _deleted: string (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- ip: string (nullable = true)
 |    |    |-- username: string (nullable = true)
 |    |-- format: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- minor: string (nullable = true)
 |    |-- model: string (nullable = true)
 |    |-- origin: long (nullable = true)
 |    |-- parentid: long (nullable = true)
 |    |-- sha1: string (nullable = true)
 |    |-- text: struct (nullable = true)
 |    |    |-- _VALUE: string (nullable = true)
 |    |    |-- _bytes: long (nullable = true)
 |    |    |-- _sha1: string (nulla

## Testing with a smaller sample of data

Creating a variable pages to hold the dataframe with selected columns, then displaying 5 rows to verify the data.


In [5]:
from pyspark.sql import functions as F

pages = (
    dataframe.select(
        F.col("title"),
        F.col("revision.text._VALUE").alias("text")
    )
    .filter(F.col("text").isNotNull())
)

In [6]:
pages

DataFrame[title: string, text: string]

## Cleaning Text field

This code block has been adapted alot to deal with issues in the resulting cleanText field generated and with wikipedia having many different features imbedded within the text itself cleaning it was a long process in terms of finding ways to use regex to keep relevant fields and remove unnecessary information.

In [23]:
#pages.count()
#pages.show(5, truncate=90)

cleanText = (
    pages
    .withColumn("cleanText", F.lower(F.col("text"))) # convert to lowercase
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\{\{[^}]*\}\}", " ")) # remove templates {{...}}
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\{\{[^}]*\}\}", " ")) # remove any nested templates {{...}}
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\[\[([^\]|]*\|)?([^\]]*)\]\]", "$2")) # remove wiki links [[...]]
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"<!--.*?-->", " "))  # remove HTML/XML comments
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"<[^>]+>", " ")) # remove HTML tags
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"<ref[^>]*>.*?</ref>", " ")) # remove references <ref>...</ref>
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\d+x\d+px", " "))  # remove image dimensions
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\b\d+px\b", " "))  # remove standalone pixel values
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\bthumb\b", " "))  # remove thumb keyword
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\balt\b", " "))  # remove alt text marker
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\bcategory\b|\bstyle\b|\bclass\b", " "))  # remove wiki metadata words
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\bright\b|\bleft\b|\bcenter\b", " "))  # remove alignment keywords
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"(?i)\.(jpg|jpeg|png|gif|svg|pdf|webp)", " "))  # remove file extensions
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"&nbsp;", " "))  # remove non-breaking spaces
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"https?://\S+", " "))  # remove URLs
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"www\.\S+", " "))  # remove www URLs
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"[^a-z\s]", " ")) # make sure only letters and spaces remain
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"'''|''", "")) # remove bold/italic markup
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"<[^>]+>", " "))  # remove all remaining HTML/XML tags
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\s+", " ")) # replace multiple spaces with a single space
    .withColumn("cleanText", F.regexp_replace(F.col("cleanText"), r"\b[b-hj-z]\b", " ")) # remove single character words excluding 'a' and 'i'
    .withColumn("cleanText", F.trim(F.col("cleanText"))) # remove any space from the beginning and end
    .filter(F.length(F.col("cleanText")) > 0) # filter out empty texts
)

## Counting Words from CleanText

Firstly, the data needs to be split into individual words, then the words can be counted up. I have displayed the top 20 words in this Wikipedia dataset.

In [24]:
from pyspark.sql import functions as F

words = (cleanText.select(
    F.explode(F.split(F.col("cleanText"), " ")).alias("word") # split the text into words
))

words = words.filter(F.col("word") != "") # filter the empty words

wordCount = words.groupBy("word").count() # counts each word

wordCount = wordCount.orderBy(F.desc("count")) # order by count descending

wordCount.show(20, truncate=False) # show the top 20 words

+-----+-------+
|word |count  |
+-----+-------+
|the  |3541653|
|of   |1931414|
|in   |1637109|
|and  |1408068|
|a    |1262410|
|to   |998791 |
|is   |897635 |
|align|606148 |
|was  |584054 |
|it   |504559 |
|for  |498282 |
|on   |473414 |
|as   |435102 |
|from |384713 |
|by   |356319 |
|that |344693 |
|he   |329229 |
|with |299470 |
|this |286644 |
|at   |271616 |
+-----+-------+
only showing top 20 rows
